# CropDetector — Phase 9, Part 3
## Train a lightweight plant-like / non-plant input validator

**Fresh, standalone Colab notebook.** It requires only the exported **`CropDetector_Phase9_Part2_Data.zip`**. No previous Colab variables, prior disease classifier checkpoint, or Google Drive mount are required.

### What Stage A can and cannot do
- **Stage A:** classify an input photograph as **plant-like (label 0)** or **non-plant (label 1)**. *Apple, squash, and unsupported tomato diseases are plant-like*.
- **Stage B (later):** identify **Maize, Grape, Potato, Tomato, or Other Crop** after Stage A.
- **Stage C (already done):** apply the existing 15-class INT8 disease classifier only after validation.

**Major limitation:** all positive examples come from PlantDoc, and all non-plant negatives come from Natural Images. The network might learn dataset-specific artifacts, not a universal notion of a plant. Development metrics will be preliminary. We must later evaluate on independently collected smartphone photographs, cluttered backgrounds, mixed scenes, unsupported crops, and blurry images. Never interpret the Stage A probability as a guaranteed safety or species test.

The notebook downloads the **same source datasets**, verifies source and manifest integrity, trains an ImageNet-pretrained MobileNetV2-0.35 at 160×160, compares a frozen-head model with a small fine-tuned model, selects a provisional development threshold, visualizes failures, and downloads a portable model/report archive. It does **not** train or change the existing 15-class disease model.

## Cell 1 — Install lightweight dependencies and initialize reproducibility

In [ ]:
%pip -q install kagglehub

import os, sys, json, io, hashlib, subprocess, shutil, time
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.metrics import (roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, roc_curve, average_precision_score)
from scipy.special import expit
import tensorflow as tf
import kagglehub
from google.colab import files

SEED = 42
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 6
TARGET_PLANT_RETENTION = 0.95
ROOT = Path('/content')
OUT = ROOT / 'CropDetector_Phase9_Part3'
OUT.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Output directory:', OUT)

## Cell 2 — Upload and validate the **Part 2 manifest ZIP**

When prompted, upload **`CropDetector_Phase9_Part2_Data.zip`**. The ZIP contains the manifests, **not the original photos**.

In [ ]:
MANIFEST_ZIP = ROOT / 'CropDetector_Phase9_Part2_Data.zip'
if not MANIFEST_ZIP.exists():
    uploaded = files.upload()
    assert MANIFEST_ZIP.name in uploaded, 'Upload CropDetector_Phase9_Part2_Data.zip'

with ZipFile(MANIFEST_ZIP) as z:
    required = {'audit_report.json', 'validator_train.csv', 'validator_dev.csv'}
    assert required.issubset(set(z.namelist())), f'Missing files: {required-set(z.namelist())}'
    audit = json.loads(z.read('audit_report.json'))
    train_df = pd.read_csv(io.BytesIO(z.read('validator_train.csv')))
    dev_df = pd.read_csv(io.BytesIO(z.read('validator_dev.csv')))

assert audit['phase'] == 'Phase9_Part2'
assert len(train_df) == audit['stage_a_train'] == 4065
assert len(dev_df) == audit['stage_a_dev'] == 1012
assert set(train_df['Target'].unique()) == {0,1}
assert set(dev_df['Target'].unique()) == {0,1}
assert set(train_df['Source'].unique()) == {'PlantDoc','NaturalImages'}
assert set(dev_df['Source'].unique()) == {'PlantDoc','NaturalImages'}
assert set(train_df['Group']).isdisjoint(set(dev_df['Group']))
assert set(train_df['SHA256']).isdisjoint(set(dev_df['SHA256']))
assert train_df['Partition'].eq('train').all()
assert dev_df['Partition'].eq('dev').all()
assert (train_df['Target'].eq(0) == train_df['Source'].eq('PlantDoc')).all()
assert (dev_df['Target'].eq(0) == dev_df['Source'].eq('PlantDoc')).all()

print('Part 2 manifest SHA-256:',hashlib.sha256(MANIFEST_ZIP.read_bytes()).hexdigest())
print('PlantDoc pinned commit:',audit['plantdoc_git_commit'])
print('Train:',len(train_df),'Dev:',len(dev_df))
display(pd.crosstab(train_df['Role'], train_df['Target']).rename_axis('Training role'))
display(pd.crosstab(dev_df['Role'], dev_df['Target']).rename_axis('Development role'))

## Cell 3 — Restore the exact PlantDoc image source

The image root is checked against the commit recorded in the manifest audit. **Do not change the source to a newer revision** without generating new manifests.

In [ ]:
PLANTDOC_ROOT = ROOT / 'PlantDoc-Dataset'
COMMIT = audit['plantdoc_git_commit']
REPO = 'https://github.com/pratikkayal/PlantDoc-Dataset.git'

if not (PLANTDOC_ROOT / '.git').exists():
    subprocess.run(['git','clone','--depth','1',REPO,str(PLANTDOC_ROOT)],check=True)

head = subprocess.check_output(['git','-C',str(PLANTDOC_ROOT),'rev-parse','HEAD'],text=True).strip()
if head != COMMIT:
    subprocess.run(['git','-C',str(PLANTDOC_ROOT),'fetch','--depth','1','origin',COMMIT],check=True)
    subprocess.run(['git','-C',str(PLANTDOC_ROOT),'checkout','--detach',COMMIT],check=True)

head = subprocess.check_output(['git','-C',str(PLANTDOC_ROOT),'rev-parse','HEAD'],text=True).strip()
assert head == COMMIT, f'PlantDoc git commit mismatch: {head} != {COMMIT}'
print('PlantDoc restored at pinned commit',head)

## Cell 4 — Re-download Natural Images and resolve exact files

The manifest contains **both** `natural_images/...` and `data/natural_images/...` path forms; resolve each relative to the dataset root exactly as supplied, without stripping directories.

In [ ]:
NATURAL_ROOT = Path(kagglehub.dataset_download('prasunroy/natural-images'))
print('Natural Images root:',NATURAL_ROOT)


def source_path(row):
    base = PLANTDOC_ROOT if row.Source=='PlantDoc' else NATURAL_ROOT
    return base / str(row.RelativePath)

for df in (train_df, dev_df):
    df['ResolvedPath'] = df.apply(source_path,axis=1).astype(str)

missing = []
for df in (train_df,dev_df):
    missing.extend(df.loc[~df.ResolvedPath.map(lambda p:Path(p).is_file()),
                          ['Source','RelativePath']].to_dict('records'))
assert not missing, (f'{len(missing)} original images not found. This may indicate a '
                     f'Kaggle dataset revision or wrong source root. First examples: {missing[:6]}')

print('All manifest paths found:',len(train_df)+len(dev_df))

## Cell 5 — Verify image fingerprints and group isolation

Verify each file against its recorded SHA-256 so that the training run uses the **same images** that were audited. For ~5,000 photographs this can take some time.

In [ ]:
def file_hash(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

all_df=pd.concat([train_df,dev_df],ignore_index=True)
verified=[]
wrong=[]
for i, r in enumerate(all_df.itertuples(index=False),start=1):
    if file_hash(r.ResolvedPath).lower() != str(r.SHA256).lower():
        wrong.append((r.Source,r.RelativePath))
    if i%1000==0:print('Verified',i,'/',len(all_df))
assert not wrong, f'Content mismatch! Do not train until fixed: {wrong[:5]}'

checks = {
    'manifest_zip_sha256':hashlib.sha256(MANIFEST_ZIP.read_bytes()).hexdigest(),
    'plantdoc_commit':COMMIT,
    'verified_images':len(all_df),
    'train_dev_shared_groups':len(set(train_df.Group)&set(dev_df.Group)),
    'train_dev_shared_sha256':len(set(train_df.SHA256)&set(dev_df.SHA256)),
}
(OUT/'data_integrity.json').write_text(json.dumps(checks,indent=2))
print('All image SHA-256 hashes verified. No group or exact-hash overlap.')

## Cell 6 — Prepare reproducible image pipelines

Images are EXIF-corrected and resized to **160×160 RGB**; the network itself rescales 0–255 to −1…1. Use exactly the same input preprocessing when deploying the validator on Android. We do not normalize pixels twice.

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE

def pil_decode_resize(path_bytes):
    path=os.fsdecode(path_bytes)
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        im=im.resize((IMG_SIZE[1],IMG_SIZE[0]),resample=Image.Resampling.BILINEAR)
        return np.asarray(im,dtype=np.uint8)

def load_image(path, target):
    x=tf.numpy_function(pil_decode_resize, [path], tf.uint8)
    x.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return tf.cast(x,tf.float32),tf.cast(target,tf.float32)

def augment(x,target):
    x=tf.image.random_flip_left_right(x)
    x=tf.image.random_brightness(x,max_delta=10.0)
    x=tf.image.random_contrast(x,lower=0.9,upper=1.1)
    return tf.clip_by_value(x,0.,255.),target

def make_dataset(frame, training=False):
    ds=tf.data.Dataset.from_tensor_slices((frame.ResolvedPath.to_numpy(dtype=str),
                                           frame.Target.to_numpy(dtype=np.float32)))
    if training:
        ds=ds.shuffle(len(frame),seed=SEED,reshuffle_each_iteration=True)
    ds=ds.map(load_image,num_parallel_calls=AUTOTUNE)
    if training:
        ds=ds.map(augment,num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds=make_dataset(train_df,training=True)
dev_ds=make_dataset(dev_df,training=False)

for x,y in train_ds.take(1):
    print('Batch shapes:',x.shape,y.shape)
    print('Pixel range:',float(tf.reduce_min(x)),float(tf.reduce_max(x)))

n0=int((train_df.Target==0).sum())
n1=int((train_df.Target==1).sum())
class_weights={0:len(train_df)/(2*n0), 1:len(train_df)/(2*n1)}
print('Training weights:',class_weights)

## Cell 7 — Build a lightweight MobileNetV2-0.35 validator

ImageNet weights are downloaded by Keras on first run. The output is **one uncalibrated logit** for **Non-plant**, not a calibrated probability. A negative logit favors plant-like; a positive one favors non-plant.

In [ ]:
from tensorflow.keras import layers, models

backbone=tf.keras.applications.MobileNetV2(
    input_shape=(160,160,3),alpha=0.35,include_top=False,weights='imagenet'
)
backbone.trainable=False

inputs=tf.keras.Input(shape=(160,160,3),dtype=tf.float32,name='rgb_image')
x=layers.Rescaling(scale=1./127.5,offset=-1.,name='mobilenet_preprocess')(inputs)
x=backbone(x,training=False)
x=layers.GlobalAveragePooling2D(name='feature_pool')(x)
x=layers.Dropout(0.25,name='training_dropout')(x)
logits=layers.Dense(1,activation=None,name='nonplant_logit')(x)
model=models.Model(inputs,logits,name='CropDetector_StageA_Validator')


def compile_model(m,lr):
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=[tf.keras.metrics.BinaryAccuracy(threshold=0.,name='accuracy')])

compile_model(model,lr=1e-3)
model.summary()
print('Parameters:',model.count_params())

## Cell 8 — Train the classifier head (backbone frozen)

In [ ]:
HEAD_PATH=OUT/'best_stageA_head.keras'
head_callbacks=[
    tf.keras.callbacks.ModelCheckpoint(str(HEAD_PATH),save_best_only=True,monitor='val_loss',mode='min'),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=2,min_lr=1e-5),
]

head_history=model.fit(train_ds,validation_data=dev_ds,
                       epochs=HEAD_EPOCHS,class_weight=class_weights,
                       callbacks=head_callbacks)
print('Saved:',HEAD_PATH)

## Cell 9 — Fine-tune only the last 25 backbone layers

Batch normalization stays frozen to avoid destabilizing pretrained statistics on this small, heterogeneous dataset. The best head checkpoint is used as the starting point.

In [ ]:
head_best=tf.keras.models.load_model(str(HEAD_PATH),compile=False)
ft_backbone=head_best.get_layer(backbone.name)
ft_backbone.trainable=True
for l in ft_backbone.layers:
    l.trainable=False
for l in ft_backbone.layers[-25:]:
    l.trainable=not isinstance(l, tf.keras.layers.BatchNormalization)

print('Trainable backbone layers:',sum(int(l.trainable) for l in ft_backbone.layers))
compile_model(head_best,lr=1e-5)
FT_PATH=OUT/'best_stageA_finetuned.keras'
ft_callbacks=[
    tf.keras.callbacks.ModelCheckpoint(str(FT_PATH),save_best_only=True,monitor='val_loss',mode='min'),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=2,restore_best_weights=True),
]
ft_history=head_best.fit(train_ds,validation_data=dev_ds,epochs=FINE_TUNE_EPOCHS,
                         class_weight=class_weights,callbacks=ft_callbacks)
print('Saved:',FT_PATH)

## Cell 10 — Compare checkpoints and evaluate **development** accuracy

These development images are from the same two source datasets as training. Scores here guide prototyping but are **not untouched real-world performance**.

In [ ]:
def predict_logits(checkpoint,dataset):
    m=tf.keras.models.load_model(str(checkpoint),compile=False)
    return m.predict(dataset.map(lambda x,y:x),verbose=0).reshape(-1)

true=dev_df.Target.to_numpy(dtype=np.int32)
checkpoints={'Frozen head':HEAD_PATH,'Fine-tuned':FT_PATH}
model_comparison=[]
logits_by_model={}

for name,path in checkpoints.items():
    scores=predict_logits(path,dev_ds)
    logits_by_model[name]=scores
    pred=(scores>=0).astype(np.int32)
    yprob=expit(scores)
    metrics={
        'Model':name,
        'Overall accuracy (%)':100*np.mean(pred==true),
        'Plant recall (%)':100*np.mean(pred[true==0]==0),
        'Non-plant recall (%)':100*np.mean(pred[true==1]==1),
        'Balanced accuracy (%)':50*(np.mean(pred[true==0]==0)+np.mean(pred[true==1]==1)),
        'AUROC (%)':100*roc_auc_score(true,scores),
        'NLL':float(np.mean(np.logaddexp(0,scores)-true*scores)),
    }
    model_comparison.append(metrics)

comparison=pd.DataFrame(model_comparison)
display(comparison.round(3))

# Choose based on smallest development negative-log-likelihood (not highest accuracy).
best_name=comparison.sort_values('NLL').iloc[0].Model
BEST_PATH=checkpoints[best_name]
BEST_MODEL=tf.keras.models.load_model(str(BEST_PATH),compile=False)
logits=logits_by_model[best_name]
print('Selected checkpoint:',best_name,BEST_PATH)
print('\nClassification report at the default logit=0 threshold:')
print(classification_report(true, (logits>=0).astype(int),
                            labels=[0,1],target_names=['Plant-like','Non-plant'],zero_division=0))
comparison.to_csv(OUT/'checkpoint_comparison.csv',index=False)

## Cell 11 — Choose a conservative plant-retention operating point

We choose the **highest plant probability threshold** that retains at least 95% of *development* plant images. The same development set is used for model selection, so reported threshold performance may be optimistic. It will be frozen and evaluated again later on independent photographs.

In [ ]:
plant_probability=expit(-logits.astype(np.float64))  # 1 - sigmoid(nonplant_logit)
plants=(true==0)
nonplants=(true==1)
plant_scores=np.sort(plant_probability[plants])
required=int(np.ceil(TARGET_PLANT_RETENTION*len(plant_scores)))
accept_threshold=float(plant_scores[len(plant_scores)-required])
accepted=plant_probability>=accept_threshold

rows=[]
for role,sub in dev_df.groupby('Role',sort=True):
    idx=sub.index.to_numpy()
    rows.append({'Role':role,'Count':len(sub),
                 'Accepted as plant':int(accepted[idx].sum()),
                 'Accepted (%)':100*np.mean(accepted[idx]),
                 'Rejected (%)':100*np.mean(~accepted[idx])})
threshold_report=pd.DataFrame(rows)

print('Plant accept threshold:',accept_threshold)
print('Development plant retention:',100*np.mean(accepted[plants]))
print('Development non-plant rejection:',100*np.mean(~accepted[nonplants]))
display(threshold_report.round(2))

obj_dev=dev_df[dev_df.Role=='Non-plant']
by_object=[]
for obj,sub in obj_dev.groupby('Folder',sort=True):
    idx=sub.index.to_numpy()
    by_object.append({'Object category':obj,'Count':len(sub),
                      'False plant acceptance (%)':100*np.mean(accepted[idx]),
                      'Rejection (%)':100*np.mean(~accepted[idx])})
object_report=pd.DataFrame(by_object)
display(object_report.round(2))
threshold_report.to_csv(OUT/'development_retention_by_role.csv',index=False)
object_report.to_csv(OUT/'development_rejection_by_object.csv',index=False)

## Cell 12 — Diagnostic plots and difficult examples

Inspect whether the network fails on mixed scenes, non-plant objects with greenery, unusual crops, or different photo styles. The figures are **diagnostic**, not a new independent test.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(plant_probability[plants],bins=25,alpha=.6,label='Plant-like (PlantDoc)',density=True)
axes[0].hist(plant_probability[nonplants],bins=25,alpha=.6,label='Non-plant (Natural Images)',density=True)
axes[0].axvline(accept_threshold,linestyle='--',color='black',label='Selected threshold')
axes[0].set_xlabel('P(plant-like)')
axes[0].set_ylabel('Density')
axes[0].legend()
fpr,tpr,_=roc_curve(true,logits)
axes[1].plot(fpr,tpr,label=f'AUROC={roc_auc_score(true,logits):.3f}')
axes[1].plot([0,1],[0,1],'--',color='gray')
axes[1].set(xlabel='Non-plant false positive rate',ylabel='Non-plant true positive rate',title='Development ROC')
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT/'stageA_distributions_and_roc.png',dpi=180)
plt.show()

# Most confidently wrong under chosen threshold
false_accept=dev_df[(true==1)&accepted].copy()
false_accept['plant_score']=plant_probability[false_accept.index]
false_reject=dev_df[(true==0)&(~accepted)].copy()
false_reject['plant_score']=plant_probability[false_reject.index]
false_accept=false_accept.sort_values('plant_score',ascending=False).head(6)
false_reject=false_reject.sort_values('plant_score',ascending=True).head(6)

fig,axes=plt.subplots(2,6,figsize=(17,6))
for row,part in enumerate([false_accept,false_reject]):
    for col,(_,r) in enumerate(part.iterrows()):
        ax=axes[row,col]
        with Image.open(r.ResolvedPath) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
        ax.set_title(f'{r.Role}: {r.Folder}\nPlant score {r.plant_score:.3f}',fontsize=9)
        ax.axis('off')
for ax in axes.flat:
    if not ax.has_data():ax.axis('off')
plt.suptitle('Top row: incorrectly accepted objects; bottom: incorrectly rejected plants')
plt.tight_layout()
plt.savefig(OUT/'stageA_failure_gallery.png',dpi=140)
plt.show()

## Cell 13 — Save model, operating threshold and provenance

The full Keras model is saved for the next phase (INT8 conversion and Android parity). The **input is floating-point RGB 0–255**; the model itself performs MobileNetV2 rescaling. Output is a **single non-plant logit**.

In [ ]:
FINAL_MODEL=OUT/'crop_validator_stageA.keras'
shutil.copy2(BEST_PATH,FINAL_MODEL)

config={
    'phase':'Phase9_Part3_StageA',
    'task':'plant-like-versus-non-plant',
    'model_architecture':'MobileNetV2 alpha=0.35 160x160 ImageNet pretrained',
    'model_file':FINAL_MODEL.name,
    'model_sha256':file_hash(FINAL_MODEL),
    'input_shape':[1,160,160,3],
    'input_dtype':'float32',
    'input_range':[0.0,255.0],
    'preprocessing':'EXIF transpose; RGB; bilinear resize 160x160; model rescales to [-1,1]',
    'output':'one logit for Non-plant',
    'labels':{'Plant-like':0,'Non-plant':1},
    'plant_score':'sigmoid(-logit)',
    'accept_if_plant_score_at_least':accept_threshold,
    'target_dev_plant_retention':TARGET_PLANT_RETENTION,
    'stageA_train_images':len(train_df),
    'stageA_dev_images':len(dev_df),
    'source_manifest_sha256':hashlib.sha256(MANIFEST_ZIP.read_bytes()).hexdigest(),
    'source_plantdoc_git_commit':COMMIT,
    'selected_checkpoint':best_name,
    'independent_field_validation':False,
    'limitations':['Dataset-source confounding: positives from PlantDoc, negatives from Natural Images',
                   'Not trained to distinguish supported crop species from unsupported crop species',
                   'Do not treat development performance as independent field reliability']
}
(OUT/'stageA_config.json').write_text(json.dumps(config,indent=2))
np.save(OUT/'development_logits.npy',logits.astype(np.float32))

training_summary={'chosen_model':best_name,
                  'model_comparison':comparison.to_dict('records'),
                  'threshold':accept_threshold,
                  'dev_plant_retention':float(np.mean(accepted[plants])),
                  'dev_nonplant_rejection':float(np.mean(~accepted[nonplants])),
                  'per_role':threshold_report.to_dict('records'),
                  'per_object':object_report.to_dict('records')}
(OUT/'training_summary.json').write_text(json.dumps(training_summary,indent=2))
print('Final Keras model:',FINAL_MODEL)
print('Input and output:',BEST_MODEL.input_shape,BEST_MODEL.output_shape)
print('Final model size MiB:',FINAL_MODEL.stat().st_size/1024**2)
print('Model SHA256:',config['model_sha256'])

## Cell 14 — Download portable Phase 9 Part 3 results

Save the ZIP on your Mac. It contains the **model**, thresholds, configuration, evaluation tables, diagnostic figures, and manifest fingerprints. The original photographs are not included. You can now close Colab without losing the trained model.

In [ ]:
archive=shutil.make_archive(str(ROOT/'CropDetector_Phase9_Part3_Results'),'zip',OUT)
print('Archive:',archive)
with ZipFile(archive) as z:
    for inf in z.infolist():
        print(f'{inf.file_size/1024**2:6.2f} MiB   {inf.filename}')
files.download(archive)

---
## Next — Part 4: Validate robustness, quantize, benchmark and add Stage B

**Do not deploy Stage A as a trustworthy plant gate solely on the numbers from this notebook.** We will review its errors; collect independent smartphone photos covering leaves, unsupported species, non-plant objects, blurry scenes, soil and clutter; evaluate the fixed checkpoint and threshold; and examine data-source bias. Then we can export INT8 and test parity on your Samsung Galaxy A14 5G. In parallel, Stage B will use the `CropLabel` field to distinguish supported crops from **Other Crop**, rather than mislabelling unsupported tomato diseases as non-plant.